# Auditing Realistic Composition Configs

We run `src.audit.audit` on three plausible-looking composition
configs and read off which structural rules each violates:

1. A naive open-r1-style stack with no gating and no differential
   cap on the LLM judge.
2. A gated but unbounded stack that lets the CoT-aware judge
   contribute arbitrarily.
3. A fully constrained stack that should pass.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from src.audit import audit, format_report
from src.composition import (
    CompositionSpec, DifferentialCap, Gate, Scorer,
)

## Config 1: open-r1-style naive stack

Inspired by the open-r1 `rewards.py` reward functions documented
at `huggingface/open-r1/blob/main/src/open_r1/rewards.py`:
`accuracy_reward`, `format_reward`, `tag_count_reward`,
`reasoning_steps_reward`, `len_reward`. We translate the stack
into the new API as faithfully as is possible without running
actual completions; in particular we treat `format_reward` and
`tag_count_reward` as additive scorers (which is how they are
wired in the GRPO config), and treat `reasoning_steps_reward` as a
CoT-reading scorer (it inspects intermediate reasoning text).

The audit will fail on R1 (CoT reader unwrapped), R2 (no gates),
and possibly R3 (cap sum vs gate penalty).

In [2]:
# Stand-in scorer functions; the actual open-r1 implementations
# operate on completion strings. The dynamics we audit are
# structural, so the toy stand-ins suffice for rule checking.
def accuracy(a): return 0.5
def fmt(a): return 0.4
def tag_count(a): return 0.3
def reasoning_steps(a): return 0.6  # this one reads CoT
def length_pen(a): return -0.1

naive_stack = CompositionSpec(
    gates=[],
    scorers=[
        Scorer(name='accuracy', fn=accuracy, cap=1.0),
        Scorer(name='format', fn=fmt, cap=1.0),
        Scorer(name='tag_count', fn=tag_count, cap=1.0),
        Scorer(name='reasoning_steps', fn=reasoning_steps,
               cap=1.0, reads_cot=True),
        Scorer(name='length_pen', fn=length_pen, cap=1.0),
    ],
    gate_penalty=-1.0,
)
report = audit(naive_stack)
print(format_report(report))

COMPOSITION AUDIT
5 components (0 gates, 5 scorers)
ok=False  seed=42

  [FAIL] differential cap on CoT readers
         CoT-reading scorers without DifferentialCap: ['reasoning_steps']. Wrap each in DifferentialCap(fn_aware, fn_blind, delta) to bound the CoT-conditioned reward signal. See docs/theory.md Section 3.

  [FAIL] gate present
         No gates defined. Format and safety prerequisites should be gates, not additive scorers. See docs/theory.md Section 1.

  [FAIL] bounded cap sum
         sum(caps)=5.000 exceeds k*|gate_penalty|=0.500 (k=0.5). A determined optimizer can pay the gate penalty in exchange for saturating every scorer.

-------------------------------------------------------
  [FAIL] differential cap on CoT readers - CoT-reading scorers without DifferentialCap: ['reasoning_steps']. Wrap each in DifferentialCap(fn_aware, fn_blind, delta) to bound the CoT-conditioned reward signal. See docs/theory.md Section 3.
  [FAIL] gate present - No gates defined. Format and saf

All three new structural rules fail: no `DifferentialCap` on the
CoT-reading `reasoning_steps`, no gates, and the cap sum exceeds
the gate magnitude bound. This is exactly the configuration that
the open-r1 community has reported produces format-reward collapse
and reward-plateau pathologies (see open-r1 issues #363 and #256).

## Config 2: gated but unbounded

A common partial fix: promote `format` to a gate. Helps with
priority inversion. Does not help with CoT drift if the LLM judge
is still unbounded.

In [3]:
gated_unbounded = CompositionSpec(
    gates=[Gate(name='format_gate', predicate=fmt, threshold=0.5)],
    scorers=[
        Scorer(name='accuracy', fn=accuracy, cap=1.0),
        Scorer(name='reasoning_steps', fn=reasoning_steps,
               cap=1.0, reads_cot=True),
        Scorer(name='length_pen', fn=length_pen, cap=1.0),
    ],
    gate_penalty=-1.0,
)
report = audit(gated_unbounded)
print(format_report(report))

COMPOSITION AUDIT
4 components (1 gates, 3 scorers)
ok=False  seed=42

  [FAIL] differential cap on CoT readers
         CoT-reading scorers without DifferentialCap: ['reasoning_steps']. Wrap each in DifferentialCap(fn_aware, fn_blind, delta) to bound the CoT-conditioned reward signal. See docs/theory.md Section 3.

  [OK  ] gate present
         1 gate(s) defined: ['format_gate'].

  [FAIL] bounded cap sum
         sum(caps)=3.000 exceeds k*|gate_penalty|=0.500 (k=0.5). A determined optimizer can pay the gate penalty in exchange for saturating every scorer.

-------------------------------------------------------
  [FAIL] differential cap on CoT readers - CoT-reading scorers without DifferentialCap: ['reasoning_steps']. Wrap each in DifferentialCap(fn_aware, fn_blind, delta) to bound the CoT-conditioned reward signal. See docs/theory.md Section 3.
  [FAIL] bounded cap sum - sum(caps)=3.000 exceeds k*|gate_penalty|=0.500 (k=0.5). A determined optimizer can pay the gate penalty in excha

R2 passes; R1 still fails because the CoT reader is not wrapped;
R3 may or may not pass depending on the cap budget.

## Config 3: fully constrained

Each CoT reader is wrapped in a `DifferentialCap` with a small
`delta`. The cap sum sits well below the gate magnitude.

In [4]:
constrained = CompositionSpec(
    gates=[Gate(name='format_gate', predicate=fmt, threshold=0.5)],
    scorers=[
        Scorer(name='accuracy', fn=accuracy, cap=0.5),
        DifferentialCap(
            name='reasoning_steps',
            fn_aware=reasoning_steps,
            fn_blind=accuracy,
            cap=0.3,
            delta=0.05,
        ),
        Scorer(name='length_pen', fn=length_pen, cap=0.2),
    ],
    gate_penalty=-2.0,
)
report = audit(constrained)
print(format_report(report))
assert report['ok']

COMPOSITION AUDIT
4 components (1 gates, 3 scorers)
ok=True  seed=42

  [OK  ] differential cap on CoT readers
         All CoT-reading scorers are wrapped in DifferentialCap.

  [OK  ] gate present
         1 gate(s) defined: ['format_gate'].

  [OK  ] bounded cap sum
         sum(caps)=1.000 within k*|gate_penalty|=1.000.



All three new rules pass. The audit is wired as a pytest fixture
(`src.audit.assert_audit_passes`), so any notebook or experiment
config that regresses will fail CI.